In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("..")
from modeling import COVARIATES, bh_fdr, oof_predict, prediction_metrics

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
TRAIN_DIR = DATA_DIR / "train_pathways"

model_input = pd.read_csv(TRAIN_DIR / "model_input_diet_cov_functional_pathways_baseline.csv", low_memory=False)
summary = pd.read_csv(TRAIN_DIR / "functional_pathway_summary_baseline.csv")
pathways = summary.loc[summary["kept_for_modeling"], "pathway"].tolist()
diet_features = [c for c in model_input.columns if c not in ["participant_id", *COVARIATES, *summary["pathway"]]]
models = {"covariates_only": COVARIATES, "covariates_plus_diet": COVARIATES + diet_features}

In [ ]:
rows = []
for target in pathways:
    y = model_input[target].to_numpy(dtype=float)
    for model, features in models.items():
        rows.append({"pathway": target, "model": model, "n": len(y),
                     **prediction_metrics(y, oof_predict(model_input[features], y))})
results = pd.DataFrame(rows)
results["fdr_q"] = results.groupby("model")["pearson_p"].transform(bh_fdr)
results["delta_r"] = (results["pearson_r"] - results.groupby("pathway")["pearson_r"].transform("first")).where(results["model"] == "covariates_plus_diet")
results.to_csv(TRAIN_DIR / "model_results_diet_predict_functional_pathways_baseline.csv", index=False)
print(results.groupby("model")[["pearson_r", "r2"]].median())